# Text Classification Model to Predict Sentence Categories
--------------------------------------------------------------------------------

## **Introduction**:
In the era of digital information, vast amounts of text data are generated every day from various sources such as social media, e-commerce websites, educational platforms, healthcare systems, and more. This unstructured text data often contains noise in the form of URLs, emojis, special characters, and irrelevant information. To derive meaningful insights from this data, it is essential to classify the text into relevant categories. This project focuses on developing an Automated Text Classification Model capable of predicting sentence categories from noisy and diverse text data.

The text data provided is unlabeled, and the categories to be predicted include:
Education,Ecommerce.Technology,Healthcare,Entertainment,Finance,News,Travel,Sports,Other.The goal is to clean the text, remove unnecessary information, and apply various machine learning models to predict the correct category with high accuracy.

--------------------------------------------------------------------------------
 ## **Objective**:
 The objective of this project is to:  
*   **Data Preprocessing and Cleaning:** Remove URLs, emojis, punctuation, special characters, and stopwords.
Normalize the text by converting it to lowercase and removing noise.

*   **Data Annotation:** Use a keyword-based approach and manual inspection to auto-label the text into one of the 10 predefined categories.

*   **Model Training and Evaluation:** Apply multiple machine learning models such as Logistic Regression, Naive Bayes, Random Forest, and Support Vector Machine (SVM).
Evaluate the performance of each model based on accuracy, precision and recall.
*   **Model Optimization and Selection:** Select the best-performing model with the highest accuracy and save the trained model for future predictions.
*   **Prediction and Deployment:** Build a separate script to load the saved model and predict the category of new text data.

--------------------------------------------------------------------------------
Importing all necessary libraries
--------------------------------------------------------------------------------

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
import pickle

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset.csv to dataset.csv


**Loading and store dataset in data variable**

In [ ]:
data = pd.read_csv("dataset.csv")

In [ ]:
data.head()

,text
0,Turn on the profile picture guard to make your...
1,►►►hier klicken: http://bit.ly/freiheitsdressu...
2,"Weekend deal alert! Outdo Santa, today only, w..."
3,THIS TEENAGE GIRL SHARES THE BIGGEST SECRET OF...
4,Easy & convenient access to professional guida...


**Define keyword-based rules for auto-labeling**

In [ ]:
category_keywords = {
    "Ecommerce": [
        "buy", "sale", "discount", "offer", "shop", "deal",
        "stores", "rate", "biscuit", "lego", "wall stickers", "alert",
        "toys", "click", "price", "shopping", "cart", "coupon", "voucher",
        "gift", "save", "purchase", "order", "delivery", "shipping",
        "furniture", "ikea", "wayfair", "walmart", "pottery barn", "crate barrel",
        "overstock", "blackfridaysale", "dyfurniture", "nocreditneeded",
        "nomoneyneeded", "whywait", "shipping", "free", "first order",
        "collect", "coins", "lightning link", "heart vegas"
    ],
    "Education": [
        "learn", "course", "study", "degree", "certification",
        "mentorship", "hbcu", "entrepreneurs", "connection", "guidance",
        "professional", "university", "college", "academy", "skills",
        "masterclass", "enroll", "internship", "training", "syllabus"
    ],
    "Technology": [
        "AI", "tech", "gadgets", "software", "computing",
        "att", "hbcuunlimited", "picture", "profile", "secret",
        "device", "data", "internet", "cloud", "smart", "innovation",
        "robotics", "security", "digital", "blockchain", "machine learning",
        "lg signature", "twinwash", "air purifier", "washing machine",
        "high-speed server", "wordpress", "optimized", "fios", "download games"
    ],
    "Travel": [
        "flight", "destination", "hotel", "vacation", "trip",
        "river", "serbia", "experience", "poppy strudel", "access",
        "tour", "cruise", "explore", "journey", "adventure", "escape",
        "booking", "ticket", "sightseeing", "resort", "getaway"
    ],
    "Entertainment": [
        "movie", "show", "series", "episode", "streaming",
        "concert", "performance", "festival", "music", "live",
        "watch", "subscribe", "comedy", "actor", "theatre",
        "celebrity", "ticket", "entertainment", "award", "drama",
        "john luskey", "babes boys tavern", "southern maryland music",
        "watch through rain view", "air purifier", "smash hit game",
        "gaten matarazzo"
    ],
    "Health & Fitness": [
        "health", "fitness", "exercise", "gym", "nutrition",
        "diet", "weight", "workout", "yoga", "trainer",
        "wellness", "calories", "protein", "vitamins", "supplement",
        "run", "cardio", "meditation", "therapy", "mental health"
    ],
    "Lifestyle": [
        "fashion", "style", "beauty", "makeup", "clothes",
        "jewelry", "home decor", "furniture", "DIY", "accessories",
        "trend", "luxury", "shoes", "brand", "designer",
        "self-care", "wellness", "living", "hobby", "culture",
        "scrambling", "switch decor", "minimalist timepieces",
        "home appliance", "season decor", "surprise"
    ],
    "Finance": [
        "money", "investment", "stocks", "trading", "savings",
        "loans", "insurance", "mutual funds", "EMI", "cryptocurrency",
        "account", "finance", "banking", "wallet", "budget",
        "interest", "credit", "tax", "pension", "payment",
        "pitchforcannes", "disruptive ideas", "adoption", "wybi",
        "practical", "feasible"
    ],
    "Food & Drink": [
        "recipe", "cooking", "baking", "dining", "restaurant",
        "snack", "dessert", "meal", "kitchen", "flavors",
        "taste", "drink", "beverage", "wine", "coffee",
        "tea", "chocolate", "alcohol", "barbecue", "chef"
    ],
    "Other": [
        "women", "child", "lamp", "moon", "surprise", "common",
        "life", "hier", "click", "truly", "weekend", "secret",
        "event", "festival", "community", "news", "announcement",
        "charity", "fundraising", "donate", "volunteer", "awareness",
        "collect now", "breathe", "ventrix", "pregnancy", "stories",
        "switch decor", "dont go broke", "ayurvedic", "ministry of ayush",
        "high-speed server", "minimalist", "timepieces", "decor", "furniture"
    ]
}


**Function for autolabeling the text**

In [ ]:
def assign_category(text):
    for category, keywords in category_keywords.items():
        if any(word in text for word in keywords):
            return category
    return "Other"

**Text preprocessing while removing stopwords,special characters and ascii chars etc**

In [ ]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
data["text"] = data["text"].fillna("").astype(str)
import unicodedata
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKD", text)
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


**Appling text cleaning**

In [ ]:
data["cleaned_text"] = data["text"].apply(clean_text)
data["category"] = data["cleaned_text"].apply(assign_category)

In [ ]:
# Save preprocessed data
data.to_csv("preprocessed_data.csv", index=False)

In [ ]:
data.head()

,text,cleaned_text,category
0,Turn on the profile picture guard to make your...,turn profile picture guard make profile pictur...,Technology
1,►►►hier klicken: http://bit.ly/freiheitsdressu...,hier klicken click,Ecommerce
2,"Weekend deal alert! Outdo Santa, today only, w...",weekend deal alert outdo santa today toys truly,Ecommerce
3,THIS TEENAGE GIRL SHARES THE BIGGEST SECRET OF...,teenage girl shares biggest secret life waht s...,Technology
4,Easy & convenient access to professional guida...,easy convenient access professional guidance r...,Education


**Defining Features and Target Variables for Text Classification**

In [ ]:
X = data["cleaned_text"]
y = data["category"]

**Splitting Data into Training and Testing Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Vectorizing Text Data Using TF-IDF for Feature Extraction**

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

**Saving TF-IDF Vectorizer for Future Use**

In [ ]:
with open("tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(tfidf_vectorizer, file)

**Comparing Model Performance and Selecting the Best Model**

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel="linear", C=1, random_state=42),
}

# Model performance comparison
best_model = None
best_accuracy = 0

for model_name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")

    print(f"Model: {model_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}\n")

    # Saving the best model based on accuracy
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model

Model: Naive Bayes
Accuracy: 0.6990
Precision: 0.7210
Recall: 0.6990

Model: Random Forest
Accuracy: 0.8817
Precision: 0.8829
Recall: 0.8817

Model: Logistic Regression
Accuracy: 0.8647
Precision: 0.8673
Recall: 0.8647

Model: SVM
Accuracy: 0.8754
Precision: 0.8737
Recall: 0.8754



**Save the best model**

In [ ]:
with open("best_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print(f"Best model saved as 'best_model.pkl' with accuracy: {best_accuracy}")

Best model saved as 'best_model.pkl' with accuracy: 0.8817


**Generating 10 sample predictions from the test set**

In [ ]:
sample_test_data = X_test.sample(10, random_state=42)
sample_predictions = best_model.predict(tfidf_vectorizer.transform(sample_test_data))

# Save sample predictions
sample_results = pd.DataFrame({"Text": sample_test_data, "Predicted Category": sample_predictions})
sample_results.to_csv("sample_predictions.csv", index=False)
print("Saving 10 sample predictions from model!")

Saving 10 sample predictions from model!


**Loading Saved Model and Vectorizer for Text Classification Prediction**

In [ ]:
# Load the saved best model
with open("best_model.pkl", "rb") as file:
    model = pickle.load(file)

# Load the vectorizer
with open("tfidf_vectorizer.pkl", "rb") as file:
    vectorizer = pickle.load(file)

# Define prediction function
def predict_category(text):
    text_transformed = vectorizer.transform([text])
    prediction = model.predict(text_transformed)[0]
    return prediction

# Example prediction
sample_text = "Enroll in AI and ML certification courses to boost your career."
predicted_category = predict_category(sample_text)
print(f"Predicted Category: {predicted_category}")

Predicted Category: Education




---


# Conclusion
After evaluating the performance of multiple models, it is evident that the Random Forest Classifier is the best-performing model for text classification in this project, with an accuracy of 88.17%. It also shows a high precision of 88.29% and recall of 88.17%, making it the most reliable model for future predictions. SVM follows closely with an accuracy of 87.54%, precision of 87.37%, and recall of 87.54%, indicating strong performance but slightly behind Random Forest. Logistic Regression also provides promising results with an accuracy of 86.47%, precision of 86.73%, and recall of 86.47%, making it a viable option for simpler models. On the other hand, Naive Bayes achieved an accuracy of 69.90%, precision of 72.10%, and recall of 69.90%, showing limitations in handling high-dimensional feature spaces generated by TF-IDF.

Considering the accuracy, precision, and recall, the Random Forest Classifier is selected as the best model. This model will be utilized for future predictions and deployment in production environments. Moving forward, the model will be saved, and an interface or API will be developed to allow real-time text classification. Additionally, continuous monitoring and fine-tuning with new data will ensure that the model's performance remains optimal over time.






---

